# Experiment 2: Ablation Study

Verify that each SoftGear component contributes independently.

**Prerequisites**: Runtime > Change runtime type > **GPU (T4)**

| Condition | Change | Tests |
|-----------|--------|-------|
| Full SoftGear | — | Baseline |
| -identity init | Random init for new gears | Initial preservation |
| -gradual (→ none) | All layers same LR (= Exp1 none) | Gradual vs no hardening |
| -gradual (→ binary) | Old/new binary LR (= Exp1 binary) | Gradual vs binary hardening |
| Decay rate 0.3/0.5/0.7 | lr_decay sweep | Hardening speed sensitivity |
| Gears 4/8/12 | Phase count | Depth effect |
| Small model | hidden_dim=64, ffn_dim=256 | Scale generalization |

**Note**: The "→ none" and "→ binary" ablations reuse Exp1 checkpoints.

## Setup

In [ ]:
!pip install -q git+https://github.com/byExist/softgear.git

In [ ]:
!softgear download sudoku9

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU not available! Change runtime type."
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# type: ignore
from google.colab import drive

drive.mount("/content/drive")
CKPT_ROOT = "/content/drive/MyDrive/softgear/checkpoints"

## Training

In [ ]:
COMMON = "--task sudoku9 --hidden-dim 128 --ffn-dim 512 --num-heads 4 --patience 5 --max-total-steps 40000 --max-samples 50000 --seed 42"

### Ablation: Identity Init

In [ ]:
# Full SoftGear (baseline, same as exp1-gradual)
!softgear train --hardening gradual --checkpoint-dir {CKPT_ROOT}/exp2-full $COMMON

In [ ]:
# Without identity init
!softgear train --hardening gradual --no-identity-init --checkpoint-dir {CKPT_ROOT}/exp2-no-identity $COMMON

### Decay Rate Sweep

In [ ]:
!softgear train --hardening gradual --lr-decay 0.3 --checkpoint-dir {CKPT_ROOT}/exp2-decay03 $COMMON

In [ ]:
!softgear train --hardening gradual --lr-decay 0.5 --checkpoint-dir {CKPT_ROOT}/exp2-decay05 $COMMON

In [ ]:
!softgear train --hardening gradual --lr-decay 0.7 --checkpoint-dir {CKPT_ROOT}/exp2-decay07 $COMMON

### Phase Count

In [ ]:
!softgear train --hardening gradual --num-gears 4 --checkpoint-dir {CKPT_ROOT}/exp2-gears4 $COMMON

In [ ]:
!softgear train --hardening gradual --num-gears 8 --checkpoint-dir {CKPT_ROOT}/exp2-gears8 $COMMON

In [ ]:
!softgear train --hardening gradual --num-gears 12 --checkpoint-dir {CKPT_ROOT}/exp2-gears12 $COMMON

### Model Scale

In [ ]:
!softgear train --hardening gradual --hidden-dim 64 --ffn-dim 256 --num-heads 4 --checkpoint-dir {CKPT_ROOT}/exp2-small \
    --task sudoku9 --patience 5 --max-total-steps 40000 --max-samples 50000 --seed 42

## Results

In [ ]:
import torch
import pandas as pd
from typing import Any
from softgear.tasks.registry import get_task
from softgear.config import DataConfig, ModelConfig

task = get_task("sudoku9")
data_cfg = DataConfig(path="data/sudoku-extreme", batch_size=64, num_workers=2, max_samples=None, curriculum=False)
_, val_loader = task.build_loaders(data_cfg)
device = torch.device("cuda")

experiments = {
    "full": f"{CKPT_ROOT}/exp2-full",
    "-identity_init": f"{CKPT_ROOT}/exp2-no-identity",
    "-gradual (→ none)": f"{CKPT_ROOT}/exp1-none",
    "-gradual (→ binary)": f"{CKPT_ROOT}/exp1-binary",
    "decay=0.3": f"{CKPT_ROOT}/exp2-decay03",
    "decay=0.5": f"{CKPT_ROOT}/exp2-decay05",
    "decay=0.7": f"{CKPT_ROOT}/exp2-decay07",
    "gears=4": f"{CKPT_ROOT}/exp2-gears4",
    "gears=8": f"{CKPT_ROOT}/exp2-gears8",
    "gears=12": f"{CKPT_ROOT}/exp2-gears12",
    "small": f"{CKPT_ROOT}/exp2-small",
}

results: dict[str, Any] = {}
for name, ckpt_dir in experiments.items():
    ckpt = torch.load(f"{ckpt_dir}/best.pt", map_location=device, weights_only=False)
    cfg = ckpt["config"]

    model_cfg = ModelConfig(**cfg["model"])
    model = task.build_model(model_cfg)
    task.mount_all_gears(model, model_cfg)
    model.load_state_dict(ckpt["model_state_dict"])
    model.to(device)
    model.eval()

    all_preds: list[torch.Tensor] = []
    all_targets: list[torch.Tensor] = []
    all_inputs: list[torch.Tensor] = []
    with torch.no_grad():
        for batch in val_loader:
            x, y = batch[0].to(device), batch[1].to(device)
            preds = task.predict_fn(model(x).logits)
            all_preds.append(preds)
            all_targets.append(y)
            all_inputs.append(x)

    metrics = task.metrics_fn(torch.cat(all_preds), torch.cat(all_targets), torch.cat(all_inputs))
    results[name] = {
        "phase": ckpt["phase"],
        "global_step": ckpt.get("global_step", "?"),
        "val_loss": ckpt["best_val_loss"],
        **metrics,
    }

df = pd.DataFrame(results).T
df.index.name = "condition"
df